## Проблема

Имеются наблюдения за бизонами: особенности поведения, физические характеристики, показатели активности. Однако определить точный возраст бизона напрямую практически сложно — животные улетают при виде человека.

Вес бизона — один из важных признаков, по которому можно судить о его возрасте. 

## Задача ML

**Задача:** предсказать возраст летающего бизона по его физическим характеристикам и показателям активности.

**Объект:** летающий бизон (одна запись в датасете).

**Образ выходного результата модели:** число — возраст бизона (возраст в годах, непрерывная величина).

**Тип задачи:** регрессия.

# Описание исходных данных

## Семантика и ожидаемые типы признаков

| Признак | Семантика | Ожидаемый тип | Целевой |
|---|---|---|---|
| Type | Тип бизона (исторические данные / дикие / прирученные) | categorical (string) | Нет |
| Length | Длина бизона | float | Нет |
| Diameter | Диаметр бизона | float | Нет |
| Height | Высота бизона | float | Нет |
| Whole_weight | Общий вес бизона | float | Нет |
| Age | Возраст бизона (в годах) | float | **Да** |


## Формулы предметной области

Формулы предметной области для предсказания возраста в датасете не предоставлены. МЕМО даёт только подсказку о возможной двухэтапной схеме: сначала предсказать вес по признакам, затем возраст по весу. Конкретные формулы связи веса и возраста не указаны — они будут определяться моделью ML на этапе обучения.

In [41]:
import pandas as pd
import numpy as np
import typing as t
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# 1. Предварительная обработка данных

## 1.1. Чтение и загрузка данных
Загружаем датасет с характеристиками летающих бизонов. Проверяем корректность загрузки: количество строк и столбцов, типы данных, наличие пропусков.


In [42]:
import os

home_dir = os.path.expanduser('~')
data_path = os.path.join(home_dir, 'OneDrive', 'Desktop', 'Studies 2026', 'ProgPractice', 'materials', 'datasets', '05_avatar.csv')

df = pd.read_csv(data_path, sep=',')

print(f"Размер датасета: {df.shape}")
print(f"\nПервые 5 строк:")
df.head()

Размер датасета: (4177, 6)

Первые 5 строк:


,Type,Length,Diameter,Height,Whole_weight,Age
0,исторические данные,22.75,18.25,9.5,514.0,15.0
1,исторические данные,17.50,13.25,9.0,225.5,7.0
2,прирученные бизоны,26.50,21.00,13.5,677.0,9.0
3,исторические данные,22.00,18.25,12.5,516.0,10.0
4,дикие бизоны,NaN,12.75,NaN,205.0,7.0


### Удаление строк с пропусками в таргете
Поскольку Age — целевой признак, строки с пропусками в нём нужно удалить. Удаляем до разделения на train/test, чтобы избежать смещения распределений.

In [43]:
df = df.dropna(subset=['Age'])

print(f"Размер после удаления пропусков в Age: {df.shape}")

Размер после удаления пропусков в Age: (3968, 6)


## 1.2. Разделение выборки на обучение и тест

In [44]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=seed)

print(f"Train размер: {train_df.shape}")
print(f"Test размер: {test_df.shape}")

Train размер: (3174, 6)
Test размер: (794, 6)


## 1.3. Первичный анализ данных

Проводим анализ только на train-выборке, чтобы избежать утечки данных. Проверяем:
- Типы данных 
- Количество уникальных значений
- Наличие пропусков в каждом признаке
- Основные статистики 


In [45]:
train_df.info()

<class 'pandas.DataFrame'>
Index: 3174 entries, 990 to 3345
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Type          3174 non-null   str    
 1   Length        2752 non-null   float64
 2   Diameter      2856 non-null   float64
 3   Height        2684 non-null   float64
 4   Whole_weight  3174 non-null   float64
 5   Age           3174 non-null   float64
dtypes: float64(5), str(1)
memory usage: 173.6 KB


In [46]:
train_df.nunique()

Type               3
Length           130
Diameter         107
Height            51
Whole_weight    2079
Age               25
dtype: int64

In [47]:
train_df.describe()

,Length,Diameter,Height,Whole_weight,Age
count,2752.000000,2856.000000,2684.000000,3174.000000,3174.000000
mean,26.212391,20.389443,13.989195,828.013863,9.915249
std,5.977469,4.962152,4.333372,491.368132,3.220551
min,3.750000,2.750000,0.000000,2.000000,1.000000
25%,22.500000,17.250000,11.500000,439.000000,8.000000
50%,27.250000,21.125000,14.000000,796.500000,9.000000
75%,30.750000,24.000000,16.500000,1154.000000,11.000000
max,40.750000,32.500000,113.000000,2779.500000,29.000000


In [48]:
print("Уникальные значения Type:")
print(train_df['Type'].value_counts())

Уникальные значения Type:
Type
исторические данные    1166
дикие бизоны           1020
прирученные бизоны      988
Name: count, dtype: int64
